# Chunking strategies

Chunking is the most underrated knob in RAG. The same corpus + the same retriever can swing recall@k by tens of points purely on how you split text. We compare four strategies on the canonical corpus:

1. **Fixed-size** (character window) — fast, dumb.
2. **Recursive** (split on paragraph → sentence → word boundaries) — the safe default.
3. **Semantic** (split where adjacent-sentence embeddings diverge) — preserves topic boundaries.
4. **Propositional** (extract atomic claims) — sketched out as a pattern; needs an LLM in production.

All four use no LLM calls in this notebook — the embedding-based semantic splitter uses the deterministic hashing embedder.

In [1]:
# %% Notebook bootstrap — never touches API keys directly.
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# CI / offline mode: replay cached responses instead of calling out.
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


LLM_CACHE_ONLY = 1


In [2]:
import re, numpy as np
from shared.embedders import hash_embed
from shared.loaders import load_corpus

DOCS = load_corpus()
SAMPLE = DOCS[1].abstract  # HA-Retrieve abstract — multi-sentence
print(SAMPLE)

Standard dense retrievers degrade on documents longer than their training context. We propose HA-Retrieve, a two-stage encoder that first scores fixed-size sentence windows with a small encoder, then routes the top-32 windows through a larger cross-encoder that produces a single document embedding. On a corpus of long scientific reports averaging 480K tokens, HA-Retrieve improves recall@10 by 17 points over a flat BGE-large baseline and reduces index size by 6x. Latency at query time is held constant by precomputing window embeddings offline. We release a 50K-document benchmark of long-form legal and biomedical text.


## 1) Fixed-size chunking
Slide a window of `size` chars with `overlap` chars; never look at structure.

In [3]:
def chunk_fixed(text: str, size: int = 200, overlap: int = 50) -> list[str]:
    step = size - overlap
    return [text[i : i + size] for i in range(0, max(1, len(text) - overlap), step)]

fixed = chunk_fixed(SAMPLE)
for i, c in enumerate(fixed):
    print(i, repr(c[:80]), '...')

0 'Standard dense retrievers degrade on documents longer than their training contex' ...
1 '-size sentence windows with a small encoder, then routes the top-32 windows thro' ...
2 'On a corpus of long scientific reports averaging 480K tokens, HA-Retrieve improv' ...
3 'ndex size by 6x. Latency at query time is held constant by precomputing window e' ...


## 2) Recursive chunking
Split on a hierarchy of separators (paragraphs → sentences → words). The safest default — you usually get something that respects sentence boundaries without an LLM in the loop.

In [4]:
SEPARATORS = ['\n\n', '\n', '. ', ' ']

def chunk_recursive(text: str, target: int = 200, seps: list[str] = SEPARATORS) -> list[str]:
    if len(text) <= target or not seps:
        return [text]
    sep, rest = seps[0], seps[1:]
    pieces = text.split(sep)
    out, buf = [], ''
    for p in pieces:
        candidate = (buf + sep + p) if buf else p
        if len(candidate) <= target:
            buf = candidate
        else:
            if buf:
                out.append(buf)
            if len(p) <= target:
                buf = p
            else:
                out.extend(chunk_recursive(p, target, rest))
                buf = ''
    if buf:
        out.append(buf)
    return out

rec = chunk_recursive(SAMPLE, target=160)
for i, c in enumerate(rec):
    print(i, repr(c[:80]), '...')

0 'Standard dense retrievers degrade on documents longer than their training contex' ...
1 'We propose HA-Retrieve, a two-stage encoder that first scores fixed-size sentenc' ...
2 'cross-encoder that produces a single document embedding' ...
3 'On a corpus of long scientific reports averaging 480K tokens, HA-Retrieve improv' ...
4 'by 6x' ...
5 'Latency at query time is held constant by precomputing window embeddings offline' ...


## 3) Semantic chunking
Embed each sentence, then split where the cosine distance between adjacent sentences exceeds a percentile threshold. Boundaries land where the *topic* shifts, not where the character count says to.

In [5]:
def split_sentences(text: str) -> list[str]:
    # Lightweight splitter; replace with a real NLP library for production text.
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

def chunk_semantic(text: str, percentile: float = 75.0) -> list[str]:
    sents = split_sentences(text)
    if len(sents) <= 1:
        return sents
    vecs = hash_embed(sents, dims=256)
    # Adjacent cosine distance (vectors are L2-normed -> distance = 1 - dot).
    dists = 1.0 - np.sum(vecs[:-1] * vecs[1:], axis=1)
    threshold = float(np.percentile(dists, percentile))
    out, current = [], [sents[0]]
    for s, d in zip(sents[1:], dists):
        if d > threshold:
            out.append(' '.join(current))
            current = [s]
        else:
            current.append(s)
    out.append(' '.join(current))
    return out

sem = chunk_semantic(SAMPLE, percentile=60)
for i, c in enumerate(sem):
    print(i, repr(c[:80]), '...')

0 'Standard dense retrievers degrade on documents longer than their training contex' ...
1 'We propose HA-Retrieve, a two-stage encoder that first scores fixed-size sentenc' ...
2 'We release a 50K-document benchmark of long-form legal and biomedical text.' ...


## 4) Propositional chunking (sketch)
Each chunk is a single atomic claim — "X causes Y", "the value of Z is N". This requires an LLM call per source paragraph to extract propositions. We don't run the LLM call here, but the pattern is straightforward:

```text
system: Extract every atomic factual claim from the text. Return one claim per line.
user: <paragraph>
```

Propositional chunking wins on faithfulness (each retrieved chunk asserts exactly one thing) at the cost of an LLM call per document during indexing. Use it when answer-correctness matters more than indexing cost.

## Compare

Run all four strategies across the corpus and measure mean / std / max chunks per document. (`propositional` is approximated by treating each sentence as its own chunk so the comparison is fair.)

In [6]:
import statistics

def chunk_propositional_proxy(text: str) -> list[str]:
    return split_sentences(text)

strategies = {
    'fixed': lambda t: chunk_fixed(t, size=240, overlap=40),
    'recursive': lambda t: chunk_recursive(t, target=240),
    'semantic': lambda t: chunk_semantic(t, percentile=60),
    'propositional*': chunk_propositional_proxy,
}
rows = []
for name, fn in strategies.items():
    counts = [len(fn(d.abstract)) for d in DOCS]
    sizes = [len(c) for d in DOCS for c in fn(d.abstract)]
    rows.append({
        'strategy': name,
        'avg_chunks_per_doc': round(statistics.mean(counts), 2),
        'avg_chunk_chars': round(statistics.mean(sizes), 1),
        'max_chunk_chars': max(sizes),
    })
import json
print(json.dumps(rows, indent=2))

[
  {
    "strategy": "fixed",
    "avg_chunks_per_doc": 3.2,
    "avg_chunk_chars": 219.7,
    "max_chunk_chars": 240
  },
  {
    "strategy": "recursive",
    "avg_chunks_per_doc": 3.8,
    "avg_chunk_chars": 160.3,
    "max_chunk_chars": 240
  },
  {
    "strategy": "semantic",
    "avg_chunks_per_doc": 2.3,
    "avg_chunk_chars": 266.8,
    "max_chunk_chars": 522
  },
  {
    "strategy": "propositional*",
    "avg_chunks_per_doc": 4.3,
    "avg_chunk_chars": 142.2,
    "max_chunk_chars": 241
  }
]


## Picking a strategy

| Strategy | Best when | Worst when |
|---|---|---|
| Fixed | Speed matters; corpus is uniform prose | Boundaries fall mid-sentence; retrieval suffers |
| Recursive | Safe default for prose | Structured docs (code, tables) need custom seps |
| Semantic | Topics shift within a document | Single-topic docs — overhead with no win |
| Propositional | Faithfulness > ingestion cost | Corpus too large to afford one LLM call per chunk |

See `eval.py` in this folder for the snapshot we commit.